# Case Study 160: Optimal Discount Prediction
## 1. Data Generation

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

n_samples = 5000
categories = ['Electronics', 'Clothing', 'Home', 'Beauty', 'Sports']
seasons = ['Spring', 'Summer', 'Autumn', 'Winter']

product_category = np.random.choice(categories, n_samples)
original_price = np.random.uniform(10, 1000, n_samples)
proposed_discount = np.random.uniform(0, 0.7, n_samples)
historical_sales = np.random.uniform(50, 5000, n_samples)
customer_demand = np.random.uniform(0.1, 1.0, n_samples)
season = np.random.choice(seasons, n_samples)
promotion_status = np.random.choice([0, 1], n_samples)
competitor_price = original_price * np.random.uniform(0.8, 1.2, n_samples)

base_sales = historical_sales * customer_demand
discount_effect = 1 + (proposed_discount * 2.5) - (proposed_discount ** 2)
competitor_effect = (competitor_price / original_price)
promotion_effect = 1 + (promotion_status * 0.3)
seasonal_effect = np.where(season == 'Winter', 1.2, np.where(season == 'Summer', 1.1, 1.0))
cat_effect_dict = {'Electronics': 1.1, 'Clothing': 1.0, 'Home': 0.9, 'Beauty': 1.05, 'Sports': 0.95}
category_effect = np.array([cat_effect_dict[c] for c in product_category])

expected_sales = (base_sales * discount_effect * competitor_effect * promotion_effect * seasonal_effect * category_effect)
noise = np.random.normal(0, expected_sales * 0.1, n_samples)
expected_sales = np.maximum(0, expected_sales + noise)

df = pd.DataFrame({
    'Product_Category': product_category,
    'Original_Price': original_price,
    'Proposed_Discount': proposed_discount,
    'Historical_Sales': historical_sales,
    'Customer_Demand': customer_demand,
    'Season': season,
    'Promotion_Status': promotion_status,
    'Competitor_Price': competitor_price,
    'Expected_Sales': expected_sales
})

df.head()

## 2. Model Training and Comparison

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X = df.drop('Expected_Sales', axis=1)
y = df['Expected_Sales']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

numeric_features = ['Original_Price', 'Proposed_Discount', 'Historical_Sales', 
                   'Customer_Demand', 'Promotion_Status', 'Competitor_Price']
categorical_features = ['Product_Category', 'Season']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first'), categorical_features)
    ])

models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=42)
}

results = {}
for name, model in models.items():
    pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    
    results[name] = {'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R2': r2}
    print(f"--- {name} ---")
    print(f"MAE: {mae:.2f} | MSE: {mse:.2f} | RMSE: {rmse:.2f} | R2: {r2:.4f}\n")

## 3. Simulating Optimal Discount for a Product

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

best_model = Pipeline(steps=[('preprocessor', preprocessor), ('model', XGBRegressor(n_estimators=100, random_state=42))])
best_model.fit(X_train, y_train)

# Example product
sample_product = pd.DataFrame({
    'Product_Category': ['Electronics'],
    'Original_Price': [500.0],
    'Historical_Sales': [2000.0],
    'Customer_Demand': [0.8],
    'Season': ['Winter'],
    'Promotion_Status': [1],
    'Competitor_Price': [480.0]
})

discount_range = np.arange(0, 0.75, 0.05) 
expected_sales = []
estimated_revenue = []

for d in discount_range:
    temp_input = sample_product.copy()
    temp_input['Proposed_Discount'] = d
    proj_sales = best_model.predict(temp_input)[0]
    
    expected_sales.append(proj_sales)
    estimated_revenue.append(proj_sales * (500.0 * (1 - d)))

plt.figure(figsize=(10, 5))
plt.plot(discount_range * 100, estimated_revenue, marker='o', color='green')
plt.title('Estimated Revenue vs Proposed Discount')
plt.xlabel('Discount (%)')
plt.ylabel('Estimated Revenue ($)')
plt.grid(True)
plt.show()

optimal_idx = np.argmax(estimated_revenue)
print(f"Optimal Discount: {discount_range[optimal_idx] * 100}%")
print(f"Maximum Revenue: ${estimated_revenue[optimal_idx]:.2f}")